# Day 5 — Lag x liquidity x participation: execution timing audit

Scope: month-end formation, execution starting only after a lag of L business days, names worked in liquidity order under a daily participation cap. Output is **timing only** (when each name gets done), no execution cost, no intraday, no clustering, no NAV. AUM 1000億, lag grid 0-20bd, cap grid 2-30%.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

DATA_DIR = Path("/home/ubuntu/work/data")
OUT_DIR  = Path("/home/ubuntu/work/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)
UNIVERSE_FILE  = DATA_DIR / "jp_top500_universe_201001_202607.parquet"
RETURNS_FILE   = DATA_DIR / "daily_returns_20100104_20260729.parquet"
PORTFOLIO_FILE = DATA_DIR / "jp_top500_monthly_bpr_portfolios_201001_202606.csv"

COLS = dict(ret_date="base_ymd", ret_code="stock_code", ret_value="daily_return",
            ret_price="price_close", ret_volume="volume",
            uni_date="date_eom", uni_code="code", uni_cap="mkt_cap",
            prt_date="DATE_EOM", prt_code="CODE", prt_weight="WEIGHT")
OI = ["#0072B2", "#009E73", "#E69F00", "#D55E00", "#56B4E9", "#CC79A7", "#F0E442", "#000000"]

AUM            = 100_000_000_000          # 1000億
LAG_GRID       = [0, 2, 5, 7, 10, 13, 15, 17, 20]        # business days after the month-end decision
CAP_GRID       = [0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]   # own share of total daily volume
BASE_LAG, BASE_CAP = 2, 0.10
ADV_WIN        = 20                       # trailing window for ADV and for the liquidity quintiles
MAX_EXEC_DAYS  = 500                      # give up on a name after this many bd from its own start
WAVE_WAIT_BD   = 20                       # a liquidity stage waits at most this long for the previous stage
BUCKET_NAMES   = ["Q1 illiquid", "Q2", "Q3", "Q4", "Q5 liquid"]   # index 0..4, 4 = most liquid
RNG = np.random.default_rng(7)

SYNTH = not (UNIVERSE_FILE.exists() and RETURNS_FILE.exists() and PORTFOLIO_FILE.exists())
if SYNTH:
    print(">>> data files not found: generating SYNTHETIC data (plumbing test only, numbers meaningless)")
    sdir = Path("/tmp/day5_synth"); sdir.mkdir(parents=True, exist_ok=True)
    days = pd.bdate_range("2019-01-02", periods=900); codes = np.array([str(1301 + i) for i in range(120)])
    nd, nn = len(days), len(codes)
    rets = RNG.normal(0.0003, 0.02, (nd, nn))
    prices = 1000 * np.exp(RNG.normal(0, 0.5, nn))[None, :] * np.cumprod(1 + rets, axis=0)
    advy = np.exp(RNG.uniform(np.log(8e7), np.log(3e10), nn))
    vol = advy[None, :] * np.exp(RNG.normal(0, 0.4, (nd, nn))) / prices
    pd.DataFrame(dict(base_ymd=np.repeat(days.strftime("%Y%m%d").astype(np.int64), nn), stock_code=np.tile(codes, nd),
                      daily_return=rets.ravel(), price_close=prices.ravel(), volume=vol.ravel())).to_parquet(sdir / "r.parquet")
    eoms = pd.DatetimeIndex(pd.Series(days, index=days).groupby(days.to_period("M")).max())
    dp = {d: i for i, d in enumerate(days)}; sh = np.exp(RNG.normal(17, 1.2, nn)); capm = prices * sh[None, :]
    pd.concat([pd.DataFrame(dict(date_eom=int(e.strftime("%Y%m%d")), code=codes, mkt_cap=capm[dp[e]])) for e in eoms]).to_parquet(sdir / "u.parquet")
    book = prices[0] * np.exp(RNG.normal(0, 0.3, nn)); prt = []
    for e in eoms[:-1]:
        pbr = prices[dp[e]] / book; pick = np.argsort(pbr)[:60]; w = (1 / pbr[pick]); w /= w.sum()
        prt.append(pd.DataFrame(dict(DATE_EOM=e.strftime("%Y-%m-%d"), CODE=codes[pick], WEIGHT=w)))
    pd.concat(prt).to_csv(sdir / "p.csv", index=False)
    UNIVERSE_FILE, RETURNS_FILE, PORTFOLIO_FILE = sdir / "u.parquet", sdir / "r.parquet", sdir / "p.csv"
print(f"SYNTH={SYNTH}  AUM={AUM:,}  lags={LAG_GRID}  caps={[f'{c:.0%}' for c in CAP_GRID]}")

## 1. Load and schema audit

Expected schema, taken from the Day 3 run (assert, do not assume):

| file | columns | note |
|---|---|---|
| `daily_returns_*.parquet` | `base_ymd` (int YYYYMMDD), `stock_code`, `daily_return`, `price_close`, `volume` (shares) | ~4,054 trading days x ~5,019 names, 2010-01-04 to 2026-07-29. Yen volume = `price_close * volume`; median across all names ~54.5m yen. |
| `jp_top500_universe_*.parquet` | `date_eom`, `code`, `mkt_cap` | monthly membership, used here only to define the liquidity quintiles cross-sectionally |
| `jp_top500_monthly_bpr_portfolios_*.csv` | `DATE_EOM`, `CODE`, `WEIGHT` | ~198 month-ends x ~100 names, weights renormalised to 1 per date |

Codes are normalised to string, stripped of a trailing `.0`, uppercased. Date columns that arrive as integers are parsed from their string form.

In [ ]:
def read(p: Path) -> pd.DataFrame:
    return pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)

def _dates(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s.astype(str)) if str(s.dtype).startswith(("int", "Int")) else pd.to_datetime(s)

def _codes(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.upper()

def tidy(p: Path, keys, name, dedupe=False) -> pd.DataFrame:
    dc, cc, vc = keys
    df = read(p)
    missing = [c for c in keys if c not in df.columns]
    assert not missing, f"{p.name}: missing {missing}. Present: {list(df.columns)}"
    out = pd.DataFrame({"date": _dates(df[dc]), "code": _codes(df[cc]), name: pd.to_numeric(df[vc], errors="coerce")}).dropna()
    return out.groupby(["date", "code"], as_index=False)[name].sum() if dedupe else out

_r = read(RETURNS_FILE)
print("returns file dtypes:"); print(_r.dtypes.to_string())
VOL = pd.DataFrame({"date": _dates(_r[COLS["ret_date"]]), "code": _codes(_r[COLS["ret_code"]]),
                    "yen_volume": pd.to_numeric(_r[COLS["ret_price"]], errors="coerce") * pd.to_numeric(_r[COLS["ret_volume"]], errors="coerce")}
                   ).dropna().groupby(["date", "code"], as_index=False)["yen_volume"].sum()
del _r
UNI = tidy(UNIVERSE_FILE, (COLS["uni_date"], COLS["uni_code"], COLS["uni_cap"]), "cap")
PRT = tidy(PORTFOLIO_FILE, (COLS["prt_date"], COLS["prt_code"], COLS["prt_weight"]), "weight", dedupe=True)

CAL = pd.DatetimeIndex(np.sort(VOL["date"].unique()))
assert len(CAL) > 100 and CAL.is_monotonic_increasing
print(f"\ncalendar : {len(CAL)} trading days, {CAL.min().date()} -> {CAL.max().date()}")
print(f"volume   : {VOL['code'].nunique()} names, median daily yen volume {VOL['yen_volume'].median():,.0f}")
print(f"universe : {UNI['date'].nunique()} month-ends x {UNI.groupby('date').size().median():.0f} names")
print(f"portfolio: {PRT['date'].nunique()} month-ends x {PRT.groupby('date').size().median():.0f} names, "
      f"weight sum per date min/max {PRT.groupby('date')['weight'].sum().min():.4f}/{PRT.groupby('date')['weight'].sum().max():.4f}")
assert (PRT["weight"] >= 0).all(), "negative portfolio weights: long-only assumption broken"
assert (VOL["yen_volume"] >= 0).all()

## 2. Panels: volume, trailing ADV, liquidity quintiles

- `VOLMAT[t, i]` yen volume, `CUMV0` its cumulative sum with a leading zero row. All completion times below are read off `CUMV0` in closed form, so there is no day loop and no possibility of a name trading before its own start index.
- `ADVMAT[t, i]` = median yen volume over the trailing `ADV_WIN` days, **shifted one day**, so the ADV used at a decision date only uses data strictly before it.
- Liquidity quintiles are cut **cross-sectionally at each month-end** on the universe members' trailing ADV, then every portfolio name is assigned to a bucket against those cut points. Bucket 4 = most liquid. A name with no ADV history gets bucket 0 (most conservative). Buckets are re-formed monthly, so a name migrating across liquidity regimes over 16 years changes bucket.

In [ ]:
CODES = np.array(sorted(set(PRT["code"]) | set(UNI["code"])))
CODES = np.array([c for c in CODES if c in set(VOL["code"])])
CPOS  = pd.Series(np.arange(len(CODES)), index=CODES); NC = len(CODES)
DPOS  = pd.Series(np.arange(len(CAL)), index=CAL)

VOLW   = VOL[VOL["code"].isin(set(CODES))].pivot_table(index="date", columns="code", values="yen_volume", aggfunc="first").reindex(CAL).reindex(columns=CODES)
VOLMAT = np.nan_to_num(VOLW.to_numpy(dtype=float), nan=0.0)
CUMV0  = np.asfortranarray(np.vstack([np.zeros((1, NC)), np.cumsum(VOLMAT, axis=0)]))   # (T+1, NC)
ADVMAT = VOLW.where(VOLW > 0).rolling(ADV_WIN, min_periods=5).median().shift(1).to_numpy(dtype=float)

def snap(dates) -> pd.DatetimeIndex:
    pos = CAL.get_indexer(pd.DatetimeIndex(pd.to_datetime(dates)), method="ffill")
    assert (pos >= 0).all(), "a date falls before the start of the trading calendar"
    return CAL[pos]

def weight_vectors(w: pd.DataFrame) -> dict:
    out = {}
    for d, g in w.groupby("date"):
        v = np.zeros(NC)
        j = CPOS.reindex(g["code"]).to_numpy(dtype=float); ok = ~np.isnan(j)
        v[j[ok].astype(int)] = g["weight"].to_numpy()[ok]
        if v.sum() > 0:
            out[snap([d])[0]] = v / v.sum()
    return out

WVEC  = weight_vectors(PRT)
FORMS = pd.DatetimeIndex(sorted(WVEC.keys()))

UNI_D = pd.DatetimeIndex(np.sort(UNI["date"].unique()))
UNI_MEM = {d: CPOS.reindex(g["code"]).dropna().to_numpy(dtype=int) for d, g in UNI.groupby("date")}
def universe_at(f):
    k = UNI_D.searchsorted(f, side="right") - 1
    return UNI_MEM[UNI_D[k]] if k >= 0 else np.arange(NC)

BUCKET = {}
for f in FORMS:
    adv = ADVMAT[int(DPOS[f])]
    mem = universe_at(f); mem = mem[np.isfinite(adv[mem]) & (adv[mem] > 0)]
    b = np.zeros(NC, dtype=int)
    if mem.size >= 25:
        cuts = np.quantile(adv[mem], [0.2, 0.4, 0.6, 0.8])
        ok = np.isfinite(adv) & (adv > 0)
        b[ok] = np.searchsorted(cuts, adv[ok], side="right")
    BUCKET[f] = np.clip(b, 0, 4)

f = FORMS[len(FORMS) // 2]
adv, bk = ADVMAT[int(DPOS[f])], BUCKET[f]
print(f"liquidity quintiles at {f.date()} (trailing {ADV_WIN}d median yen volume):")
for q in range(5):
    sel = (bk == q) & np.isfinite(adv)
    print(f"  {BUCKET_NAMES[q]:<12} n={sel.sum():4d}  median ADV {np.nanmedian(adv[sel]):>18,.0f}")
print(f"\npanels: {VOLMAT.shape[0]} days x {NC} names; {len(FORMS)} formation dates")

## 4. Timing engine

One rule, applied per name: on any day the fund may take at most `cap` of that day's **total** volume including its own print, i.e. at most `cap/(1-cap)` of the printed market volume. So a target trade of `X` yen needs cumulative market volume of `X*(1-cap)/cap` from the day it starts. Completion day is therefore read straight off the cumulative volume column with a binary search, no day loop, which also makes it structurally impossible for a name to trade before its start index.

Ordering (`order`):

- `liquid_first` (the task convention): Q5 starts at decision + L; each next quintile starts the business day after the previous quintile has fully completed, or after `WAVE_WAIT_BD` days, whichever comes first, so one stuck illiquid tail cannot stall the schedule indefinitely.
- `parallel`: every name starts at decision + L (Day 3 behaviour, control).
- `illiquid_first`: reverse waves (control; this is the order that minimises total completion time, so it is the natural benchmark for what liquid-first costs in calendar time).

Trades are the raw month-over-month weight change `w_new - w_old` times AUM, undrifted, consistent with Day 3.

In [ ]:
def shift_bd_idx(f, n: int):
    i = int(DPOS[snap([f])[0]]) + n
    return i if 0 <= i < len(CAL) else None

def completion_idx(need_vol, cols, start_idx, max_exec=MAX_EXEC_DAYS):
    """Smallest calendar index t >= start_idx at which cumulative market volume since start_idx
    covers need_vol.
      t >= 0 : completed on that day
      -1     : the cap is genuinely binding, not done within max_exec days of a full window
      -2     : the allowed window runs past the end of the sample, so we cannot tell (truncated)"""
    out = np.full(len(cols), -1, dtype=int)
    truncated = start_idx + max_exec > len(CAL) - 1
    lim = min(start_idx + max_exec, len(CAL))
    for k in range(len(cols)):
        col = CUMV0[:, cols[k]]
        idx = int(np.searchsorted(col, col[start_idx] + need_vol[k], side="left"))
        t = idx - 1
        if idx <= len(CAL) - 1 and start_idx <= t < lim:
            out[k] = t
        elif truncated:
            out[k] = -2
    return out

def month_paths(delta_w, bucket_of, i_dec, lag, cap, order="liquid_first",
                max_exec=MAX_EXEC_DAYS, wave_wait=WAVE_WAIT_BD):
    act = np.flatnonzero(delta_w != 0)
    s0 = i_dec + lag
    if act.size == 0 or s0 >= len(CAL):
        return None
    need = np.abs(delta_w[act]) * AUM * (1.0 - cap) / cap
    bk = bucket_of[act]
    start = np.full(act.size, s0, dtype=int); finish = np.full(act.size, -1, dtype=int)
    if order == "parallel":
        groups = [np.arange(act.size)]
    else:
        seq = [4, 3, 2, 1, 0] if order == "liquid_first" else [0, 1, 2, 3, 4]
        groups = [np.flatnonzero(bk == b) for b in seq]
    s = s0
    for g in groups:
        if g.size == 0:
            continue
        start[g] = s
        fin = completion_idx(need[g], act[g], s, max_exec)
        finish[g] = fin
        done = fin >= 0
        stage_end = int(fin[done].max()) if done.any() else min(s + max_exec, len(CAL) - 1)
        s = int(min(stage_end + 1, s + wave_wait, len(CAL) - 1))
    return act, start, finish, bk, need

def run_combo(lag, cap, order="liquid_first", detail=False):
    """Runs every month. Returns per-bucket timing stats; detail=True also returns the name-month frame."""
    parts = []
    for m in range(1, len(FORMS)):
        d1 = FORMS[m]; i_dec = int(DPOS[d1])
        res = month_paths(WVEC[d1] - WVEC[FORMS[m - 1]], BUCKET[d1], i_dec, lag, cap, order)
        if res is None:
            continue
        act, start, finish, bk, need = res
        parts.append(pd.DataFrame({
            "decision": d1, "code": CODES[act], "bucket": bk,
            "start_offset_bd": start - i_dec,
            "days_exec": np.where(finish >= 0, finish - start + 1, np.nan),
            "days_from_decision": np.where(finish >= 0, finish - i_dec + 1, np.nan),
            "trade_yen": need * cap / (1.0 - cap),
            "truncated": finish == -2,
        }))
    D_all = pd.concat(parts, ignore_index=True)
    trunc = D_all.groupby("bucket")["truncated"].mean().reindex(range(5))
    D = D_all[~D_all["truncated"]].copy()          # sample-end cases carry no information about capacity
    g = D.groupby("bucket")
    S = pd.DataFrame({
        "n": g.size(),
        "mean_start_offset_bd": g["start_offset_bd"].mean(),
        "mean_days_exec": g["days_exec"].mean(),
        "median_days_exec": g["days_exec"].median(),
        "p90_days_exec": g["days_exec"].quantile(0.90),
        "mean_days_from_decision": g["days_from_decision"].mean(),
        "median_days_from_decision": g["days_from_decision"].median(),
        "share_unresolved": g["days_exec"].apply(lambda s: s.isna().mean()),
        "share_truncated_by_sample_end": trunc,
        "median_trade_yen": g["trade_yen"].median(),
    }).reindex(range(5))
    S.index = pd.Index([BUCKET_NAMES[b] for b in S.index], name="bucket")
    S.insert(0, "cap", cap); S.insert(0, "lag_bd", lag); S.insert(0, "order", order)
    return (S, D_all) if detail else S

## 3. Lag logic audit

The task convention: the portfolio is decided on month-end information, and **nothing may trade before that month-end**. Execution then starts L business days later, working the most liquid quintile first.

Checks:

- **A1** every formation date is the last trading day of its month.
- **A2** `shift_bd(f, L)` lands exactly L trading days after `f` in the calendar index.
- **A3** no name's start index is earlier than `index(f) + L`, for every month, every lag, every order.
- **A4** under liquid-first ordering the mean start offset is monotonically non-decreasing from Q5 to Q1.
- **A5** completion duration is weakly decreasing in the participation cap, within each bucket.
- **A6** a worked example: one month printed line by line, so the dates can be eyeballed.

In [ ]:
# A1 formation dates are the last trading day of their month
eom_of_month = pd.Series(CAL, index=CAL).groupby(CAL.to_period("M")).max()
bad_eom = [f for f in FORMS if eom_of_month.get(f.to_period("M")) != f]
print(f"A1 formation dates that are NOT the month's last trading day: {len(bad_eom)}" + ("" if not bad_eom else f"  {bad_eom[:5]}"))

# A2 lag arithmetic is exact in trading days
a2 = all(int(DPOS[CAL[shift_bd_idx(f, L)]]) - int(DPOS[snap([f])[0]]) == L
         for f in FORMS[:-1] for L in LAG_GRID if shift_bd_idx(f, L) is not None)
print(f"A2 shift_bd_idx lands exactly L trading days after the decision: {a2}")

# A3 nothing starts before decision + L, for every month / lag / order
viol = 0
for order in ["liquid_first", "parallel", "illiquid_first"]:
    for L in LAG_GRID:
        for m in range(1, len(FORMS)):
            d1 = FORMS[m]; i_dec = int(DPOS[d1])
            r = month_paths(WVEC[d1] - WVEC[FORMS[m - 1]], BUCKET[d1], i_dec, L, 0.30, order)
            if r is None:
                continue
            viol += int((r[1] < i_dec + L).sum())
print(f"A3 name-months starting before decision + lag: {viol}  (must be 0)")
assert viol == 0

# A4 liquid-first really does start the liquid quintile first
S_lf = run_combo(BASE_LAG, BASE_CAP, "liquid_first")
so = S_lf["mean_start_offset_bd"].reindex(BUCKET_NAMES[::-1])
print(f"A4 mean start offset (bd from decision), Q5 -> Q1: {so.round(2).to_dict()}")
print(f"   non-decreasing from liquid to illiquid: {bool(np.all(np.diff(so.dropna().to_numpy()) >= -1e-9))}")

# A5 more capacity -> weakly faster, within each bucket
mono = run_combo(BASE_LAG, 0.02)["mean_days_exec"], run_combo(BASE_LAG, 0.30)["mean_days_exec"]
print(f"A5 mean exec days at cap 2% vs 30%:")
print(pd.DataFrame({"cap 2%": mono[0], "cap 30%": mono[1]}).round(1).to_string())

# A6 one month, printed
m = len(FORMS) // 2; d0, d1 = FORMS[m - 1], FORMS[m]; i_dec = int(DPOS[d1])
act, start, finish, bk, need = month_paths(WVEC[d1] - WVEC[FORMS[m - 1]], BUCKET[d1], i_dec, BASE_LAG, BASE_CAP)
ex = pd.DataFrame({"code": CODES[act], "bucket": [BUCKET_NAMES[b] for b in bk],
                   "trade_yen": need * BASE_CAP / (1 - BASE_CAP),
                   "start_date": CAL[start], "finish_date": [CAL[t] if t >= 0 else pd.NaT for t in finish],
                   "start_offset_bd": start - i_dec,
                   "days_exec": [t - s + 1 if t >= 0 else np.nan for t, s in zip(finish, start)]})
print(f"\nA6 worked example  decision (month-end) {d1.date()}  ->  earliest allowed trade date {CAL[i_dec + BASE_LAG].date()}  (lag {BASE_LAG}bd, cap {BASE_CAP:.0%})")
print(ex.sort_values(["bucket", "trade_yen"], ascending=[False, False]).groupby("bucket", observed=True).head(2).to_string(index=False))
print("\nstage start dates by quintile:")
print(ex.groupby("bucket", observed=True)["start_date"].min().sort_index(ascending=False).to_string())

## 5. Base case: how long each liquidity bucket takes

Base combination is lag 2bd, cap 10%, liquid-first. Three views: the per-bucket table, the aggregate fill trajectory (share of the month's target trade completed by day h after the decision), and the ordering comparison.

In [ ]:
S_base, D_base = run_combo(BASE_LAG, BASE_CAP, "liquid_first", detail=True)
print(f"base case: AUM {AUM:,.0f}, lag {BASE_LAG}bd, cap {BASE_CAP:.0%}, liquid-first")
print(S_base.drop(columns=["order", "lag_bd", "cap"]).round(2).to_string())

# fill trajectory: share of target trade value completed by day h after the decision
H = np.arange(0, 61)
fill = np.zeros((5, len(H))); tot = np.zeros(5)
for m in range(1, len(FORMS)):
    d1 = FORMS[m]; i_dec = int(DPOS[d1])
    r = month_paths(WVEC[d1] - WVEC[FORMS[m - 1]], BUCKET[d1], i_dec, BASE_LAG, BASE_CAP)
    if r is None:
        continue
    act, start, finish, bk, need = r
    f_ = BASE_CAP / (1 - BASE_CAP)
    tgt = need * f_
    tt = np.minimum(i_dec + H + 1, len(CAL))
    got = np.minimum(tgt[None, :], (CUMV0[tt][:, act] - CUMV0[start, act][None, :]) * f_)
    got = np.where((i_dec + H)[:, None] >= start[None, :], np.maximum(got, 0.0), 0.0)
    for q in range(5):
        sel = bk == q
        if sel.any():
            fill[q] += got[:, sel].sum(axis=1)
            tot[q] += tgt[sel].sum()
FILL = pd.DataFrame((fill / np.where(tot > 0, tot, np.nan)[:, None]).T, index=H, columns=BUCKET_NAMES)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for q in range(5):
    ax[0].plot(H, FILL[BUCKET_NAMES[q]] * 100, color=OI[q], label=BUCKET_NAMES[q])
ax[0].axvline(BASE_LAG, lw=0.8, ls="--", color="0.4"); ax[0].set_xlabel("business days after the month-end decision")
ax[0].set_ylabel("% of target trade executed"); ax[0].set_title(f"Fill trajectory (lag {BASE_LAG}bd, cap {BASE_CAP:.0%}, liquid-first)")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.25)

ORD = pd.concat([run_combo(BASE_LAG, BASE_CAP, o) for o in ["liquid_first", "parallel", "illiquid_first"]])
piv_ord = ORD.reset_index().pivot(index="bucket", columns="order", values="mean_days_from_decision").reindex(BUCKET_NAMES)
x = np.arange(5)
for i, o in enumerate(piv_ord.columns):
    ax[1].bar(x + (i - 1) * 0.27, piv_ord[o], width=0.27, color=OI[i], label=o)
ax[1].set_xticks(x); ax[1].set_xticklabels(BUCKET_NAMES, fontsize=8)
ax[1].set_ylabel("mean bd from decision to completion"); ax[1].set_title("Ordering comparison")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.25, axis="y")
fig.tight_layout(); fig.savefig(OUT_DIR / "day5_base_case.png", dpi=150); plt.show()

print("\nmean business days from decision to completion, by ordering:")
print(piv_ord.round(1).to_string())
_lf, _pl = piv_ord.loc["Q1 illiquid", "liquid_first"], piv_ord.loc["Q1 illiquid", "parallel"]
print(f"\nliquid-first costs the illiquid quintile {_lf - _pl:+.1f} bd of extra delay versus trading everything at once, "
      f"and buys the liquid quintile {piv_ord.loc['Q5 liquid', 'parallel'] - piv_ord.loc['Q5 liquid', 'liquid_first']:+.1f} bd. "
      "Whether that trade is worth making is a cost question, not a timing one, so it is left to the next pass.")

## 6. Full grid: lag x participation cap x liquidity

`LAG_GRID` x `CAP_GRID` under liquid-first ordering, 63 combinations, five quintiles each. Two timing measures are reported and they answer different questions:

- **`days_exec`** = business days from a name's own first trading day to completion. This is pure capacity: it should depend strongly on the cap and on liquidity, and be **nearly flat in lag** (the only channel is that a different lag samples a different volume window). If it moves materially with lag, something is coupling the lag to capacity that should not be.
- **`days_from_decision`** = business days from the month-end decision to completion. This is the number that matters for implementation shortfall: lag, queue wait behind more liquid quintiles, and execution duration all land in it, and it should rise roughly one for one with lag.

In [ ]:
rows = []
for L in LAG_GRID:
    for c in CAP_GRID:
        rows.append(run_combo(L, c, "liquid_first"))
GRID = pd.concat(rows).reset_index()
GRID.to_csv(OUT_DIR / "day5_lag_cap_liquidity_grid.csv", index=False)
print(f"grid: {len(GRID)} rows -> {OUT_DIR / 'day5_lag_cap_liquidity_grid.csv'}")

for b in BUCKET_NAMES:
    print(f"\nmean days_exec (own start -> done), {b}   rows = cap, cols = lag(bd)")
    print(GRID[GRID["bucket"] == b].pivot(index="cap", columns="lag_bd", values="mean_days_exec").round(1).to_string())

fig, axes = plt.subplots(1, 5, figsize=(22, 3.9))
vmax = GRID["mean_days_exec"].max()
for q, b in enumerate(BUCKET_NAMES):
    P = GRID[GRID["bucket"] == b].pivot(index="cap", columns="lag_bd", values="mean_days_exec")
    im = axes[q].imshow(P.to_numpy(), aspect="auto", cmap="viridis_r", vmin=0, vmax=vmax)
    axes[q].set_xticks(range(len(P.columns))); axes[q].set_xticklabels(P.columns, fontsize=8)
    axes[q].set_yticks(range(len(P.index))); axes[q].set_yticklabels([f"{v:.0%}" for v in P.index], fontsize=8)
    axes[q].set_xlabel("lag (bd)"); axes[q].set_title(b, fontsize=10)
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            axes[q].text(j, i, f"{P.to_numpy()[i, j]:.0f}", ha="center", va="center", fontsize=6.5,
                         color="w" if P.to_numpy()[i, j] > vmax * 0.45 else "k")
axes[0].set_ylabel("participation cap")
fig.suptitle("Mean business days to complete, measured from the name's own start", y=1.03)
fig.tight_layout(); fig.savefig(OUT_DIR / "day5_days_exec_heatmaps.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(18, 4.3))
for q, b in enumerate(BUCKET_NAMES):
    g = GRID[(GRID["bucket"] == b) & (GRID["lag_bd"] == BASE_LAG)].sort_values("cap")
    ax[0].plot(g["cap"] * 100, g["mean_days_exec"], marker="o", color=OI[q], label=b)
    g2 = GRID[(GRID["bucket"] == b) & (GRID["cap"] == BASE_CAP)].sort_values("lag_bd")
    ax[1].plot(g2["lag_bd"], g2["mean_days_from_decision"], marker="o", color=OI[q], label=b)
    ax[2].plot(g2["lag_bd"], g2["mean_days_exec"], marker="o", color=OI[q], label=b)
ax[0].set_xlabel("participation cap (%)"); ax[0].set_ylabel("mean days_exec"); ax[0].set_yscale("log")
ax[0].set_title(f"Capacity: exec days vs cap (lag {BASE_LAG}bd)")
ax[1].set_xlabel("lag (bd)"); ax[1].set_ylabel("mean days from decision"); ax[1].set_title(f"Total delay vs lag (cap {BASE_CAP:.0%})")
ax[2].set_xlabel("lag (bd)"); ax[2].set_ylabel("mean days_exec"); ax[2].set_title("Audit: exec duration should be flat in lag")
for a in ax:
    a.legend(fontsize=7); a.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(OUT_DIR / "day5_lag_cap_lines.png", dpi=150); plt.show()

UNRES = GRID.pivot_table(index=["bucket", "cap"], columns="lag_bd", values="share_unresolved").loc[BUCKET_NAMES]
print(f"\nshare of name-months not completed within {MAX_EXEC_DAYS}bd of their own start:")
print((UNRES * 100).round(2).to_string())
fig, ax = plt.subplots(figsize=(7.5, 4))
for q, b in enumerate(BUCKET_NAMES):
    g = GRID[(GRID["bucket"] == b) & (GRID["lag_bd"] == BASE_LAG)].sort_values("cap")
    ax.plot(g["cap"] * 100, g["share_unresolved"] * 100, marker="o", color=OI[q], label=b)
ax.set_xlabel("participation cap (%)"); ax.set_ylabel(f"% unresolved within {MAX_EXEC_DAYS}bd")
ax.set_title(f"Names that never finish (AUM {AUM/1e8:,.0f} oku, lag {BASE_LAG}bd)")
ax.legend(fontsize=8); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(OUT_DIR / "day5_unresolved.png", dpi=150); plt.show()

## 7. Reading it, and what this deliberately does not do

**What the grid answers.** For a 1000億 fund running this monthly low-PBR list, how many trading days each liquidity quintile actually needs at a given participation cap, and how much calendar time sits between the month-end decision and a completed position once the lag and the liquid-first queue are added.

**Checks to make before quoting any of it:**
- `days_exec` should be close to flat across the lag axis within a bucket. A visible slope means the lag is changing available capacity, which it should not.
- `days_from_decision` minus `days_exec` minus `start_offset_bd` should be zero by construction; the start offset is where the lag and the queue wait show up separately.
- Watch `share_unresolved` in Q1 at low caps. At 1000億 a 1% weight is 10億 in one name; against a Q1 median ADV near 2億 that is many days even at a 30% cap, and at 2% it may not finish at all. If Q1 unresolved is large, the honest conclusion is that this AUM cannot hold the illiquid tail at that cap, which is a result, not a bug.

**Known simplifications, all deliberate for this pass:**
- No execution cost, no spread, no impact. Timing only.
- No intraday split and no clustering indicator. Both attach later, at the point where a day's slice gets priced.
- Trades are the undrifted month-over-month weight change; a drift-consistent fund engine changes the trade sizes slightly but not the ordering or the capacity arithmetic.
- Participation caps are applied to the same day's printed volume (mild foresight, same convention as Day 3). An ADV-based cap is a one-line change and would be the conservative variant.
- Volume is close times shares from the returns file, so it is a proxy for 売買代金, not the exchange's own figure.